In [1]:
import os
import pandas as pd
from pathlib import Path
import shutil

In [2]:
# load dataset specification files (based on Manifold Bias paper)

calibration_real = pd.read_excel("calibration_real_paths.xlsx")
test_generated = pd.read_excel("test_generated_paths.xlsx")
test_real = pd.read_excel("test_real_paths.xlsx")

## CNNSpot
filtering relevant images from CNNSpot

### Real

In [23]:
CNNSpot_real = test_real[test_real["dataset name"] == "CNNSpot"].copy()

length_CNNSpot_real = len(CNNSpot_real)

length_CNNSpot_real

26088

In [24]:
CNNSpot_real.head()

,path,dataset name
3,train2/airplane/0_real/00928.png,CNNSpot
4,train2/cow/0_real/00758.png,CNNSpot
13,train2/sofa/0_real/00370.png,CNNSpot
14,train2/bird/0_real/00033.png,CNNSpot
20,train2/bus/0_real/01214.png,CNNSpot


In [25]:
split_CNNSpot_real = CNNSpot_real["path"].str.split(pat="/", expand=True)

# drop filename (last column)
split_CNNSpot_real = split_CNNSpot_real.iloc[:, :-1]

# optionally name the levels
split_CNNSpot_real.columns = [f"level_{i}" for i in range(split_CNNSpot_real.shape[1])]

# count per level and store in dict
level_counts = {
    col: split_CNNSpot_real[col].value_counts()
    for col in split_CNNSpot_real.columns
}

# print nicely sorted by level
for level, counts in level_counts.items():
    print(f"\n=== {level} ===")
    print(counts)


=== level_0 ===
level_0
train2     24920
backup2     1168
Name: count, dtype: int64

=== level_1 ===
level_1
bottle         1317
sofa           1312
person         1311
cat            1311
sheep          1310
car            1310
bicycle        1310
tvmonitor      1309
cow            1309
horse          1307
boat           1307
airplane       1304
train          1301
dog            1301
chair          1298
diningtable    1297
bird           1297
pottedplant    1296
motorbike      1292
bus            1289
Name: count, dtype: int64

=== level_2 ===
level_2
0_real    26088
Name: count, dtype: int64


In [26]:
splitted_CNNSpot_real = CNNSpot_real["path"].str.split(pat="/")
CNNSpot_real["real_path"] = ("CNNSpot/train/" + splitted_CNNSpot_real.str[1]+ "/" + splitted_CNNSpot_real.str[2]+ "/" + splitted_CNNSpot_real.str[3])

In [27]:
root = Path("../../datasets/")

CNNSpot_real["exists"] = [
    (root / p).exists() for p in CNNSpot_real["real_path"]
]

print(CNNSpot_real["exists"].value_counts()) # results mean that all file paths exist, within the CNNSpot folder

exists
False    26088
Name: count, dtype: int64


In [28]:
duplicates_CNN_real = CNNSpot_real.real_path[CNNSpot_real.real_path.duplicated()]
(f"Number of duplicate paths: {len(duplicates_CNN_real)}")

'Number of duplicate paths: 18'

In [29]:
src_root = Path("../../datasets")      # where files currently are
dst_root = Path("../../replication_datasets")   # where you want them

for rel_path in CNNSpot_real["real_path"]:
    src = src_root / rel_path
    dst = dst_root / rel_path

    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)  # create folders
        shutil.copy2(src, dst)  # preserves metadata
    else:
        print(f"Missing: {src}")

Missing: ../../datasets/CNNSpot/train/airplane/0_real/00928.png
Missing: ../../datasets/CNNSpot/train/cow/0_real/00758.png
Missing: ../../datasets/CNNSpot/train/sofa/0_real/00370.png
Missing: ../../datasets/CNNSpot/train/bird/0_real/00033.png
Missing: ../../datasets/CNNSpot/train/bus/0_real/01214.png
Missing: ../../datasets/CNNSpot/train/motorbike/0_real/01288.png
Missing: ../../datasets/CNNSpot/train/motorbike/0_real/00575.png
Missing: ../../datasets/CNNSpot/train/boat/0_real/00367.png
Missing: ../../datasets/CNNSpot/train/cat/0_real/00246.png
Missing: ../../datasets/CNNSpot/train/train/0_real/00705.png
Missing: ../../datasets/CNNSpot/train/tvmonitor/0_real/00307.png
Missing: ../../datasets/CNNSpot/train/motorbike/0_real/01325.png
Missing: ../../datasets/CNNSpot/train/dog/0_real/00334.png
Missing: ../../datasets/CNNSpot/train/airplane/0_real/00028.png
Missing: ../../datasets/CNNSpot/train/cow/0_real/00498.png
Missing: ../../datasets/CNNSpot/train/airplane/0_real/01080.png
Missing: ../

In [30]:
root = Path("../../replication_datasets/CNNSpot")

extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"}

count = sum(1 for p in root.rglob("*") if p.suffix.lower() in extensions)

print(count)
print(length_CNNSpot_real-18) # there are 18 duplicates

94189
26070


### Fake

In [3]:
CNNSpot_fake = test_generated[test_generated["dataset name"] == "CNNSpot"].copy()

length_CNNSpot_fake = len(CNNSpot_fake)

length_CNNSpot_fake

72590

In [4]:
CNNSpot_fake.head()

,path,dataset name
0,stylegan/car/1_fake/008017.png,CNNSpot
1,train2/person/1_fake/00570.png,CNNSpot
3,cyclegan/orange/1_fake/n07740461_12101_fake.png,CNNSpot
4,train2/cow/1_fake/01321.png,CNNSpot
5,cyclegan/orange/1_fake/n07740461_14960_fake.png,CNNSpot


In [5]:
split_fake = CNNSpot_fake["path"].str.split(pat="/")

CNNSpot_fake["part0"] = split_fake.str[0]
CNNSpot_fake["part1"] = split_fake.str[1]
CNNSpot_fake["part2"] = split_fake.str[2]
CNNSpot_fake["part3"] = split_fake.str[3]

In [6]:
CNNSpot_fake

,path,dataset name,part0,part1,part2,part3
0,stylegan/car/1_fake/008017.png,CNNSpot,stylegan,car,1_fake,008017.png
1,train2/person/1_fake/00570.png,CNNSpot,train2,person,1_fake,00570.png
3,cyclegan/orange/1_fake/n07740461_12101_fake.png,CNNSpot,cyclegan,orange,1_fake,n07740461_12101_fake.png
4,train2/cow/1_fake/01321.png,CNNSpot,train2,cow,1_fake,01321.png
5,cyclegan/orange/1_fake/n07740461_14960_fake.png,CNNSpot,cyclegan,orange,1_fake,n07740461_14960_fake.png
...,...,...,...,...,...,...
100694,stylegan2/cat/1_fake/001589.png,CNNSpot,stylegan2,cat,1_fake,001589.png
100695,stylegan2/horse/1_fake/001545.png,CNNSpot,stylegan2,horse,1_fake,001545.png
100696,train2/train/1_fake/00997.png,CNNSpot,train2,train,1_fake,00997.png
100698,crn/1_fake/100212_output.png,CNNSpot,crn,1_fake,100212_output.png,NaN


In [7]:
print(CNNSpot_fake.part0.unique())
print(CNNSpot_fake.part1.unique())
print(CNNSpot_fake.part2.unique())
print(CNNSpot_fake.part3.unique())

# ! train2 and the different generators from the test split follow different path structures
# training set: train2/<obejct_name>/0_real/<file_name>
# testing set: <generator_name>/1_fake/<file_name>

['stylegan' 'train2' 'cyclegan' 'imle' 'stylegan2' 'san' 'crn' 'gaugan'
 'biggan']
['car' 'person' 'orange' 'cow' '1_fake' 'airplane' 'church' 'dog' 'bird'
 'chair' 'bottle' 'sofa' 'bicycle' 'summer' 'pottedplant' 'winter' 'sheep'
 'horse' 'cat' 'diningtable' 'motorbike' 'train' 'boat' 'tvmonitor' 'bus'
 'bedroom' 'zebra' 'apple']
['1_fake' '00100639.png' '00100807.png' ... '00100825.png' '00267139.png'
 '000000043435.png']
['008017.png' '00570.png' 'n07740461_12101_fake.png' ... '036150.png'
 '095833.png' '075918.png']


In [8]:
CNNSpot_fake["real_path"] = CNNSpot_fake["path"]

mask_train = CNNSpot_fake["part0"] == "train2"
mask_not_train = CNNSpot_fake["part0"] != "train2"

CNNSpot_fake.loc[mask_train, "real_path"] = (
    CNNSpot_fake.loc[mask_train, "path"]
    .str.replace("^train2/", "CNNSpot/train/", regex=True)
)

CNNSpot_fake.loc[mask_not_train, "real_path"] = (
    "CNNSpot/test/" + CNNSpot_fake.loc[mask_not_train, "path"]
)

In [9]:
CNNSpot_fake

,path,dataset name,part0,part1,part2,part3,real_path
0,stylegan/car/1_fake/008017.png,CNNSpot,stylegan,car,1_fake,008017.png,CNNSpot/test/stylegan/car/1_fake/008017.png
1,train2/person/1_fake/00570.png,CNNSpot,train2,person,1_fake,00570.png,CNNSpot/train/person/1_fake/00570.png
3,cyclegan/orange/1_fake/n07740461_12101_fake.png,CNNSpot,cyclegan,orange,1_fake,n07740461_12101_fake.png,CNNSpot/test/cyclegan/orange/1_fake/n07740461_...
4,train2/cow/1_fake/01321.png,CNNSpot,train2,cow,1_fake,01321.png,CNNSpot/train/cow/1_fake/01321.png
5,cyclegan/orange/1_fake/n07740461_14960_fake.png,CNNSpot,cyclegan,orange,1_fake,n07740461_14960_fake.png,CNNSpot/test/cyclegan/orange/1_fake/n07740461_...
...,...,...,...,...,...,...,...
100694,stylegan2/cat/1_fake/001589.png,CNNSpot,stylegan2,cat,1_fake,001589.png,CNNSpot/test/stylegan2/cat/1_fake/001589.png
100695,stylegan2/horse/1_fake/001545.png,CNNSpot,stylegan2,horse,1_fake,001545.png,CNNSpot/test/stylegan2/horse/1_fake/001545.png
100696,train2/train/1_fake/00997.png,CNNSpot,train2,train,1_fake,00997.png,CNNSpot/train/train/1_fake/00997.png
100698,crn/1_fake/100212_output.png,CNNSpot,crn,1_fake,100212_output.png,NaN,CNNSpot/test/crn/1_fake/100212_output.png


In [10]:
root = Path("../../datasets/")

CNNSpot_fake["exists"] = [
    (root / p).exists() for p in CNNSpot_fake["real_path"]
]

print(CNNSpot_fake["exists"].value_counts()) # results mean that all file paths exist, within the CNNSpot folder

exists
True     71535
False     1055
Name: count, dtype: int64


In [11]:
pd.set_option("display.max_colwidth", None)

In [12]:
missing = CNNSpot_fake[~CNNSpot_fake["exists"]]

missing["real_path"].head(20)

21      CNNSpot/test/cyclegan/summer/1_fake/2016-01-02 20_15_00_fake.png
28      CNNSpot/test/cyclegan/winter/1_fake/2013-07-22 00_17_40_fake.png
472     CNNSpot/test/cyclegan/winter/1_fake/2016-05-21 00_48_51_fake.png
480     CNNSpot/test/cyclegan/winter/1_fake/2015-06-29 14_59_01_fake.png
495     CNNSpot/test/cyclegan/summer/1_fake/2009-02-20 00_18_51_fake.png
519     CNNSpot/test/cyclegan/winter/1_fake/2012-06-14 05_38_40_fake.png
547     CNNSpot/test/cyclegan/summer/1_fake/2013-03-02 03_57_51_fake.png
611     CNNSpot/test/cyclegan/winter/1_fake/2013-07-12 20_32_51_fake.png
655     CNNSpot/test/cyclegan/winter/1_fake/2013-08-23 20_48_11_fake.png
892     CNNSpot/test/cyclegan/winter/1_fake/2012-05-15 07_40_41_fake.png
893     CNNSpot/test/cyclegan/winter/1_fake/2011-10-12 01_13_31_fake.png
1101    CNNSpot/test/cyclegan/summer/1_fake/2016-01-12 20_58_51_fake.png
1260    CNNSpot/test/cyclegan/summer/1_fake/2010-11-04 16_00_00_fake.png
1484    CNNSpot/test/cyclegan/winter/1_fake/2015-09

In [13]:
len(CNNSpot_fake[
    CNNSpot_fake["part1"].isin(["summer", "winter"])
])

1055

In [14]:
mask = CNNSpot_fake["part1"].isin(["summer", "winter"])

CNNSpot_fake.loc[mask, "real_path"] = CNNSpot_fake.loc[mask, "real_path"].str.replace(
    r'(\d{2})_(\d{2})_(\d{2})(_[^/]+$)',
    r'\1:\2:\3\4',
    regex=True
)

In [15]:
root = Path("../../datasets/")

CNNSpot_fake["exists"] = [
    (root / p).exists() for p in CNNSpot_fake["real_path"]
]

print(CNNSpot_fake["exists"].value_counts()) # results mean that all file paths exist, within the CNNSpot folder

exists
True    72590
Name: count, dtype: int64


In [16]:
duplicates_CNN_fake = CNNSpot_fake.real_path[CNNSpot_fake.real_path.duplicated()]
(f"Number of duplicate paths: {len(duplicates_CNN_fake)}")

'Number of duplicate paths: 4471'

In [17]:
duplicates_CNN_fake = CNNSpot_fake.path[CNNSpot_fake.path.duplicated()]
(f"Number of duplicate paths: {len(duplicates_CNN_fake)}")

'Number of duplicate paths: 4471'

In [18]:
src_root = Path("../../datasets")      # where files currently are
dst_root = Path("../../replication_datasets")   # where you want them

for rel_path in CNNSpot_fake["real_path"]:
    src = src_root / rel_path
    dst = dst_root / rel_path

    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)  # create folders
        shutil.copy2(src, dst)  # preserves metadata
    else:
        print(f"Missing: {src}")

In [22]:
root = Path("../../replication_datasets/CNNSpot")

extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"}

count = sum(1 for p in root.rglob("*") if p.suffix.lower() in extensions)

print(count)
print(length_CNNSpot_real - 18 + length_CNNSpot_fake - 4471)

94189
94189


## GenImage